In [ ]:
import kagglehub
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
# Build a custom dataset class to load images and masks
# Use Dataloaders to prepare your data
# Display some images and their corresponding masks
from PIL import Image
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import os
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import os
import numpy as np
import matplotlib.pyplot as plt

class SUIMDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, size=(256, 256)):
        self.image_paths = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.lower().endswith((".jpg", ".png"))])
        self.mask_paths = sorted([os.path.join(masks_dir, f) for f in os.listdir(masks_dir) if f.lower().endswith((".png", ".jpg"))])
        self.transform = transform or transforms.ToTensor()
        self.size = size

        if len(self.image_paths) == 0 or len(self.mask_paths) == 0:
          pass
        if len(self.image_paths) != len(self.mask_paths):
          pass
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB").resize(self.size, Image.BILINEAR)
        mask_img = Image.open(self.mask_paths[idx]).resize(self.size, Image.NEAREST)

        image = self.transform(image)

        mask = np.array(mask_img)
        if mask.ndim == 3:
            mask = mask[..., 0]
        mask = remap_mask(torch.from_numpy(mask.copy()).long())

        return image, mask

dataset_root = os.path.join(path, "dataset")
images_dir = os.path.join(dataset_root, "images")
masks_dir = os.path.join(dataset_root, "masks")

full_dataset = SUIMDataset(images_dir, masks_dir, transform=transforms.ToTensor(), size=(256, 256))

generator = torch.Generator()
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0, pin_memory=True)

imgs, masks = next(iter(train_loader))
n_show = min(4, imgs.shape[0])
fig, axes = plt.subplots(n_show, 2, figsize=(8, 3 * n_show))
if n_show == 1:
    axes = np.array([axes])

for i in range(n_show):
    axes[i, 0].imshow(imgs[i].cpu().detach().numpy().transpose(1, 2, 0))
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks[i].cpu().detach().numpy(), vmin=0, vmax=7)
    axes[i, 1].set_title("Mask")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
# Use a pretrained UNet from segmentation_models_pytorch with efficientnet-b1 as an encoder
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b0",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=1,  # Binary segmentation (1 output channel)
).to(device)

In [ ]:
# TO DO
# Define the training and validation loops

import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).to(torch.float)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).to(torch.float)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
# Define the loss and the optimizer
# Train the model
# Print the training and validation losses
# Plot loss curve
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
import torch.nn as nn

num_classes = 8

# Define U-Net Model (multi-class)
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b0",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes,
).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0.0
    for images, masks in tqdm(train_loader):
        images = images.to(device)
        masks = masks.to(device).long()

        outputs = model(images)  # [B, C, H, W]
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
    train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device).long()

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_val_loss += loss.item()
    val_loss = total_val_loss / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

plt.plot(range(1, num_epochs + 1), train_losses, label="Train Loss", marker="o")
plt.plot(range(1, num_epochs + 1), val_losses, label="Validation Loss", marker="o")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()


In [ ]:
# TO DO
# Visualize your mode's predicitons against the ground truth for several images